In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ==============================================================
# CROSS VALIDATION WITH VALIDATION ACCURACY + LOSS + CONFUSION
# ==============================================================

import matplotlib.pyplot as plt
import seaborn as sns


def cross_validation(X, y, groups):

    cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=config["random_state"]
    )

    results = []

    val_true_all = []
    val_pred_all = []

    train_acc_history = []
    val_acc_history = []

    train_loss_history = []
    val_loss_history = []


    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), 1):

        print("\n================")
        print("FOLD", fold)
        print("================")


        tr_idx, val_idx = train_test_split(
            train_idx,
            test_size=config["validation_split"],
            stratify=y[train_idx],
            random_state=42
        )


        scaler = StandardScaler()


        X_train = scaler.fit_transform(
            X[tr_idx].reshape(-1,X.shape[-1])
        ).reshape(X[tr_idx].shape)


        X_val = scaler.transform(
            X[val_idx].reshape(-1,X.shape[-1])
        ).reshape(X[val_idx].shape)



        model = build_transformer(
            X.shape[1],
            X.shape[2]
        )


        model.compile(
            optimizer=Adam(config["learning_rate"]),
            loss="binary_crossentropy",
            metrics=["accuracy"]
        )


        history = model.fit(
            X_train,
            y[tr_idx],
            validation_data=(X_val,y[val_idx]),
            epochs=config["epochs"],
            batch_size=config["batch_size"],
            verbose=1
        )


        # ============================
        # SAVE TRAINING PERFORMANCE
        # ============================

        train_acc_history.append(
            history.history["accuracy"]
        )

        val_acc_history.append(
            history.history["val_accuracy"]
        )


        train_loss_history.append(
            history.history["loss"]
        )

        val_loss_history.append(
            history.history["val_loss"]
        )



        # ============================
        # VALIDATION ACCURACY
        # ============================

        val_prob=model.predict(
            X_val,
            verbose=0
        )

        val_pred=(val_prob>=0.5).astype(int).ravel()


        val_acc=accuracy_score(
            y[val_idx],
            val_pred
        )*100


        print(
            "Validation Accuracy:",
            val_acc
        )


        print(
            classification_report(
                y[val_idx],
                val_pred,
                digits=4
            )
        )


        val_true_all.extend(
            y[val_idx]
        )

        val_pred_all.extend(
            val_pred
        )



        results.append({

            "Fold":fold,
            "Validation Accuracy":val_acc

        })


        tf.keras.backend.clear_session()
        del model
        gc.collect()



    # ==========================================================
    # MEAN VALIDATION ACCURACY
    # ==========================================================

    result_df=pd.DataFrame(results)


    print("\n======================")
    print("FINAL VALIDATION RESULT")
    print("======================")

    print(
        "Mean Validation Accuracy:",
        result_df["Validation Accuracy"].mean()
    )


    result_df.to_csv(
        f"{config['result_dir']}/validation_accuracy.csv",
        index=False
    )



    # ==========================================================
    # CONFUSION MATRIX
    # ==========================================================

    cm=confusion_matrix(
        val_true_all,
        val_pred_all
    )


    plt.figure(figsize=(7,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        annot_kws={"size":18}
    )


    plt.xlabel(
        "Predicted Label",
        fontsize=18
    )

    plt.ylabel(
        "True Label",
        fontsize=18
    )


    plt.xticks(
        fontsize=15
    )

    plt.yticks(
        fontsize=15
    )


    plt.tight_layout()


    plt.savefig(
        f"{config['result_dir']}/validation_confusion_matrix.png",
        dpi=600,
        bbox_inches="tight"
    )


    plt.close()



    # ==========================================================
    # ACCURACY CURVE
    # ==========================================================

    plt.figure(figsize=(8,6))


    plt.plot(
        np.mean(train_acc_history,axis=0),
        linewidth=3,
        label="Training Accuracy"
    )


    plt.plot(
        np.mean(val_acc_history,axis=0),
        linewidth=3,
        label="Validation Accuracy"
    )


    plt.xlabel(
        "Epoch",
        fontsize=18
    )

    plt.ylabel(
        "Accuracy",
        fontsize=18
    )


    plt.legend(
        fontsize=14
    )


    plt.grid(True)


    plt.tight_layout()


    plt.savefig(
        f"{config['result_dir']}/validation_accuracy_curve.png",
        dpi=600
    )

    plt.close()



    # ==========================================================
    # LOSS CURVE
    # ==========================================================


    plt.figure(figsize=(8,6))


    plt.plot(
        np.mean(train_loss_history,axis=0),
        linewidth=3,
        label="Training Loss"
    )


    plt.plot(
        np.mean(val_loss_history,axis=0),
        linewidth=3,
        label="Validation Loss"
    )


    plt.xlabel(
        "Epoch",
        fontsize=18
    )


    plt.ylabel(
        "Loss",
        fontsize=18
    )


    plt.legend(
        fontsize=14
    )


    plt.grid(True)


    plt.tight_layout()


    plt.savefig(
        f"{config['result_dir']}/validation_loss_curve.png",
        dpi=600
    )


    plt.close()



    return result_df

In [ ]:

# ==============================================================
# EEG TRANSFORMER CLASSIFICATION USING 5-FOLD STRATIFIED GROUP CV
# ==============================================================

import os
import re
import gc
import logging
import warnings
from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd
import mne
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization,
    MultiHeadAttention, GlobalAveragePooling1D, Layer
)
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

warnings.filterwarnings("ignore")

# ==============================================================
# CONFIGURATION
# ==============================================================  

config = {
    "data_path": "/kaggle/input/eegdatasetn/data/*.edf",  # <-- CHANGE TO YOUR PATH
    "fs": 128,
    "epoch_duration": 1,
    "embed_dim": 16,
    "num_heads": 2,
    "ff_dim": 32,
    "transformer_blocks": 2,
    "dropout": 0.3,
    "learning_rate": 0.001,
    "batch_size": 64,
    "epochs": 20,
    "validation_split": 0.15,
    "random_state": 42,
    "cache_dir": "cache",
    "result_dir": "results"
}

Path(config["cache_dir"]).mkdir(exist_ok=True)
Path(config["result_dir"]).mkdir(exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger()

# ==============================================================
# EEG CHANNELS
# ==============================================================

channels = [
    'Fp1','Fp2',
    'F7','F3','Fz','F4','F8',
    'T3','C3','Cz','C4','T4',
    'T5','P3','Pz','P4','T6',
    'O1','O2'
]

# ==============================================================
# LABEL + SUBJECT
# ==============================================================

def get_label(filename):
    name = Path(filename).stem.lower()
    if name.startswith("h"):
        return 0
    elif name.startswith("s"):
        return 1
    else:
        raise ValueError(f"Filename must start with 'h' or 's': {filename}")

def get_subject(filename):
    name = Path(filename).stem.lower()
    m = re.search(r"[hs]\d+", name)
    return m.group() if m else name

# ==============================================================
# TRANSFORMER BLOCK
# ==============================================================

class TransformerBlock(Layer):
    def __init__(self, embed_dim, heads, ff_dim, dropout):
        super().__init__()

        self.attention = MultiHeadAttention(num_heads=heads, key_dim=embed_dim)

        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="gelu"),
            Dense(embed_dim)
        ])

        self.norm1 = LayerNormalization(epsilon=1e-6)
        self.norm2 = LayerNormalization(epsilon=1e-6)

        self.drop1 = Dropout(dropout)
        self.drop2 = Dropout(dropout)

    def call(self, x, training=False):
        att = self.attention(x, x)
        att = self.drop1(att, training=training)
        out1 = self.norm1(x + att)

        ffn = self.ffn(out1)
        ffn = self.drop2(ffn, training=training)

        return self.norm2(out1 + ffn)

# ==============================================================
# MODEL
# ==============================================================

def build_transformer(time_steps, channels_num):
    inputs = Input(shape=(time_steps, channels_num))

    x = Dense(config["embed_dim"])(inputs)

    for _ in range(config["transformer_blocks"]):
        x = TransformerBlock(
            config["embed_dim"],
            config["num_heads"],
            config["ff_dim"],
            config["dropout"]
        )(x)

    x = GlobalAveragePooling1D()(x)

    x = Dense(64, activation="gelu")(x)
    x = Dropout(config["dropout"])(x)

    outputs = Dense(1, activation="sigmoid", dtype="float32")(x)

    return Model(inputs, outputs)

# ==============================================================
# EEG PROCESSOR
# ==============================================================

class EEGProcessor:
    def __init__(self):
        self.samples = int(config["fs"] * config["epoch_duration"])

    def load(self, file):
        raw = mne.io.read_raw_edf(file, preload=True, verbose=False)

        if int(raw.info["sfreq"]) != config["fs"]:
            raw.resample(config["fs"])

        raw.pick(channels)
        return raw

    def preprocess(self, raw):
        data = raw.get_data()

        n = data.shape[1] // self.samples
        if n == 0:
            return None

        data = data[:, :n * self.samples]

        data = data.reshape(len(channels), n, self.samples)
        data = np.transpose(data, (1, 2, 0))

        return data

    def process(self, file, label):
        sid = get_subject(file)

        x_cache = Path(config["cache_dir"]) / f"{sid}_x.npy"
        y_cache = Path(config["cache_dir"]) / f"{sid}_y.npy"

        if x_cache.exists() and y_cache.exists():
            try:
                X = np.load(x_cache)
                y = np.load(y_cache)
                return X, y, sid
            except Exception as e:
                print(f"  -> Cache load error for {sid}: {e}. Reprocessing.")
                # fall through

        raw = self.load(file)
        X = self.preprocess(raw)

        if X is None:
            return None

        y = np.ones(len(X)) * label

        np.save(x_cache, X)
        np.save(y_cache, y)

        return X, y, sid

# ==============================================================
# CROSS VALIDATION (ONLY VALIDATION METRICS + PLOTS)
# ==============================================================

def cross_validation(X, y, groups):

    cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=config["random_state"]
    )

    # Store per‑fold validation accuracy
    results = []

    # Collect all validation predictions and true labels across folds
    val_true_all = []
    val_pred_all = []

    # Store histories for plotting (mean across folds)
    train_acc_history = []
    val_acc_history = []
    train_loss_history = []
    val_loss_history = []

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), 1):

        print("\n================")
        print("FOLD", fold)
        print("================")

        # Split train into train + validation (we ignore the outer test_idx)
        tr_idx, val_idx = train_test_split(
            train_idx,
            test_size=config["validation_split"],
            stratify=y[train_idx],
            random_state=42
        )

        # Scale data
        scaler = StandardScaler()

        X_train = scaler.fit_transform(
            X[tr_idx].reshape(-1, X.shape[-1])
        ).reshape(X[tr_idx].shape)

        X_val = scaler.transform(
            X[val_idx].reshape(-1, X.shape[-1])
        ).reshape(X[val_idx].shape)

        # Build and compile model
        model = build_transformer(X.shape[1], X.shape[2])

        model.compile(
            optimizer=Adam(config["learning_rate"]),
            loss="binary_crossentropy",
            metrics=["accuracy"]
        )

        # Train
        history = model.fit(
            X_train,
            y[tr_idx],
            validation_data=(X_val, y[val_idx]),
            epochs=config["epochs"],
            batch_size=config["batch_size"],
            verbose=1
        )

        # Save histories
        train_acc_history.append(history.history["accuracy"])
        val_acc_history.append(history.history["val_accuracy"])
        train_loss_history.append(history.history["loss"])
        val_loss_history.append(history.history["val_loss"])

        # Evaluate on validation set
        val_prob = model.predict(X_val, verbose=0)
        val_pred = (val_prob >= 0.5).astype(int).ravel()
        val_acc = accuracy_score(y[val_idx], val_pred) * 100

        print(f"Validation Accuracy: {val_acc:.2f}%")
        print(classification_report(y[val_idx], val_pred, digits=4))

        # Collect predictions for overall report
        val_true_all.extend(y[val_idx])
        val_pred_all.extend(val_pred)

        results.append({"Fold": fold, "Validation Accuracy": val_acc})

        # Clean up
        tf.keras.backend.clear_session()
        del model
        gc.collect()

    # ==========================================================
    # 1. MEAN CLASSIFICATION REPORT (aggregated over all folds)
    # ==========================================================
    report = classification_report(
        val_true_all,
        val_pred_all,
        digits=4,
        output_dict=True
    )
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(f"{config['result_dir']}/validation_classification_report.csv")

    print("\n===== MEAN VALIDATION CLASSIFICATION REPORT =====")
    print(report_df)

    # ==========================================================
    # 2. MEAN VALIDATION ACCURACY ± STD
    # ==========================================================
    mean_acc = np.mean([r["Validation Accuracy"] for r in results])
    std_acc = np.std([r["Validation Accuracy"] for r in results])
    print(f"\nMean Validation Accuracy: {mean_acc:.2f}% ± {std_acc:.2f}%")

    # Save per‑fold accuracies
    results_df = pd.DataFrame(results)
    results_df.to_csv(f"{config['result_dir']}/validation_accuracy_per_fold.csv", index=False)

    # ==========================================================
    # 3. CONFUSION MATRIX (aggregated over all validation samples)
    # ==========================================================
    cm = confusion_matrix(val_true_all, val_pred_all)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                annot_kws={"size": 18}, cbar=False)
    plt.xlabel("Predicted Label", fontsize=18)
    plt.ylabel("True Label", fontsize=18)
    plt.xticks(fontsize=15)
    plt.yticks(fontsize=15)
    plt.tight_layout()
    plt.savefig(f"{config['result_dir']}/validation_confusion_matrix.png", dpi=600, bbox_inches="tight")
    plt.close()

    # ==========================================================
    # 4. ACCURACY CURVE (mean over folds)
    # ==========================================================
    plt.figure(figsize=(8, 6))
    plt.plot(np.mean(train_acc_history, axis=0), linewidth=3, label="Training Accuracy")
    plt.plot(np.mean(val_acc_history, axis=0), linewidth=3, label="Validation Accuracy")
    plt.xlabel("Epoch", fontsize=18)
    plt.ylabel("Accuracy", fontsize=18)
    plt.legend(fontsize=14)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"{config['result_dir']}/validation_accuracy_curve.png", dpi=600)
    plt.close()

    # ==========================================================
    # 5. LOSS CURVE (mean over folds)
    # ==========================================================
    plt.figure(figsize=(8, 6))
    plt.plot(np.mean(train_loss_history, axis=0), linewidth=3, label="Training Loss")
    plt.plot(np.mean(val_loss_history, axis=0), linewidth=3, label="Validation Loss")
    plt.xlabel("Epoch", fontsize=18)
    plt.ylabel("Loss", fontsize=18)
    plt.legend(fontsize=14)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"{config['result_dir']}/validation_loss_curve.png", dpi=600)
    plt.close()

    return results_df

# ==============================================================
# MAIN
# ==============================================================

def main():

    processor = EEGProcessor()

    files = sorted(glob(config["data_path"]))
    print(f"Found {len(files)} files matching pattern: {config['data_path']}")

    if not files:
        print("No files found! Check your data_path.")
        return

    X_all, y_all, groups = [], [], []

    for file in files:
        print(f"\nProcessing: {file}")
        try:
            label = get_label(file)
            result = processor.process(file, label)

            if result is None:
                print(f"  -> Skipping {file} (processor returned None)")
                continue

            X, y, sid = result
            if len(X) == 0:
                print(f"  -> Skipping {file} (no epochs extracted)")
                continue

            X_all.append(X)
            y_all.append(y)
            groups.extend([sid] * len(y))
            print(f"  -> Added {len(X)} epochs from subject {sid}")

        except Exception as e:
            print(f"  -> ERROR processing {file}: {e}")
            import traceback
            traceback.print_exc()

    if not X_all:
        print("\nNo data loaded. Check your files, labels, and preprocessing.")
        return

    X = np.vstack(X_all)
    y = np.hstack(y_all)
    groups = np.array(groups)

    print(f"\nEEG shape: {X.shape}, labels: {y.shape}, groups: {groups.shape}")

    # Run cross‑validation (now only validation metrics)
    df = cross_validation(X, y, groups)

    print("\n====================")
    print("FINAL RESULTS")
    print("====================")
    print(f"Mean Validation Accuracy: {df['Validation Accuracy'].mean():.2f}%")
    print(f"Std Dev: {df['Validation Accuracy'].std():.2f}%")
    print("All plots and CSVs saved in:", config["result_dir"])

if __name__ == "__main__":
    main()


